# Lab 14 - Algoritmo KNN 
## Objetivo 
Implementar el algoritmo K-Nearest Neighbors(KNN) para clasificar especies de flores utilizando el dataset Wine, aplicando el flujo completo de un modelo supervisado: preparación de datos, entrenamiento, predicción y evaluación del modelo.

In [1]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [23]:
## Descargar el datasetde Wine
wine = load_wine(as_frame=True)

print(wine.DESCR)

.. _wine_dataset:

Wine recognition dataset
------------------------

**Data Set Characteristics:**

:Number of Instances: 178
:Number of Attributes: 13 numeric, predictive attributes and the class
:Attribute Information:
    - Alcohol
    - Malic acid
    - Ash
    - Alcalinity of ash
    - Magnesium
    - Total phenols
    - Flavanoids
    - Nonflavanoid phenols
    - Proanthocyanins
    - Color intensity
    - Hue
    - OD280/OD315 of diluted wines
    - Proline
    - class:
        - class_0
        - class_1
        - class_2

:Summary Statistics:

============================= ==== ===== ======= =====
                                Min   Max   Mean     SD
============================= ==== ===== ======= =====
Alcohol:                      11.0  14.8    13.0   0.8
Malic Acid:                   0.74  5.80    2.34  1.12
Ash:                          1.36  3.23    2.36  0.27
Alcalinity of Ash:            10.6  30.0    19.5   3.3
Magnesium:                    70.0 162.0    99.7  14.3

## Crear el DataFrame

In [3]:
dfWine = wine.frame

dfWine.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


## Explorar el dataset

In [4]:
dfWine.shape

(178, 14)

In [5]:
dfWine.columns

Index(['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium',
       'total_phenols', 'flavanoids', 'nonflavanoid_phenols',
       'proanthocyanins', 'color_intensity', 'hue',
       'od280/od315_of_diluted_wines', 'proline', 'target'],
      dtype='object')

In [6]:
dfWine.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    float64
 13  targe

In [7]:
dfWine.describe()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
count,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000
mean,13.000618,2.336348,2.366517,19.494944,99.741573,2.295112,2.029270,0.361854,1.590899,5.058090,0.957449,2.611685,746.893258,0.938202
std,0.811827,1.117146,0.274344,3.339564,14.282484,0.625851,0.998859,0.124453,0.572359,2.318286,0.228572,0.709990,314.907474,0.775035
min,11.030000,0.740000,1.360000,10.600000,70.000000,0.980000,0.340000,0.130000,0.410000,1.280000,0.480000,1.270000,278.000000,0.000000
25%,12.362500,1.602500,2.210000,17.200000,88.000000,1.742500,1.205000,0.270000,1.250000,3.220000,0.782500,1.937500,500.500000,0.000000
50%,13.050000,1.865000,2.360000,19.500000,98.000000,2.355000,2.135000,0.340000,1.555000,4.690000,0.965000,2.780000,673.500000,1.000000
75%,13.677500,3.082500,2.557500,21.500000,107.000000,2.800000,2.875000,0.437500,1.950000,6.200000,1.120000,3.170000,985.000000,2.000000
max,14.830000,5.800000,3.230000,30.000000,162.000000,3.880000,5.080000,0.660000,3.580000,13.000000,1.710000,4.000000,1680.000000,2.000000


In [8]:
dfWine.isnull().sum()

alcohol                         0
malic_acid                      0
ash                             0
alcalinity_of_ash               0
magnesium                       0
total_phenols                   0
flavanoids                      0
nonflavanoid_phenols            0
proanthocyanins                 0
color_intensity                 0
hue                             0
od280/od315_of_diluted_wines    0
proline                         0
target                          0
dtype: int64

## Modificar el nombre de la columna

In [9]:
dfWine.rename(columns={"target":"clase"}, inplace=True)

X = dfWine.drop("clase", axis=1)
y = dfWine["clase"]

## Variables predictoras y objetivo

In [10]:
X = dfWine.drop(columns=["clase"])

print("Variables predictoras")
print(X.columns)

Variables predictoras
Index(['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium',
       'total_phenols', 'flavanoids', 'nonflavanoid_phenols',
       'proanthocyanins', 'color_intensity', 'hue',
       'od280/od315_of_diluted_wines', 'proline'],
      dtype='object')


In [11]:
y = dfWine["clase"]

print("Variable objetivo")
print(y.name)

Variable objetivo
clase


## Comenzar con la separación de datos de entrenamiento y de prueba 

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [13]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (142, 13)
X_test: (36, 13)
y_train: (142,)
y_test: (36,)


## Realizar el escalado de los datos 

In [14]:
scaler = StandardScaler()

X_train_escalado = scaler.fit_transform(X_train)
X_test_escalado = scaler.transform(X_test)

In [15]:
print("Datos originales")
print(X_train.head())

print("\nDatos escalados")
print(X_train_escalado[:5])

Datos originales
     alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
36     13.28        1.64  2.84               15.5      110.0           2.60   
30     13.73        1.50  2.70               22.5      101.0           3.00   
26     13.39        1.77  2.62               16.1       93.0           2.85   
12     13.75        1.73  2.41               16.0       89.0           2.60   
148    13.32        3.24  2.38               21.5       92.0           1.93   

     flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
36         2.68                  0.34             1.36             4.60  1.09   
30         3.25                  0.29             2.38             5.70  1.19   
26         2.94                  0.34             1.45             4.80  0.92   
12         2.76                  0.29             1.81             5.60  1.15   
148        0.76                  0.45             1.25             8.42  0.55   

     od280/od315_of_d

## Comenzamos con la creación del modelo KNN

In [16]:
modelo = KNeighborsClassifier(n_neighbors=3)

## Realizamos el entrenamiento

In [17]:
modelo.fit(
    X_train_escalado,
    y_train
)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",3
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](3,)","[0,1,2]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


## Hacemos predicciones

In [18]:
prediccion = modelo.predict(X_test_escalado)

print(prediccion[:10])

[0 2 0 0 1 0 0 1 1 2]


In [19]:
comparacion = pd.DataFrame({
    "Clase real": y_test.values,
    "Clase predicha": prediccion
})

comparacion.head(15)

,Clase real,Clase predicha
0,0,0
1,2,2
2,0,0
3,1,0
4,1,1
5,0,0
6,0,0
7,1,1
8,1,1
9,2,2


## Se evalúa la exactitud del modelo

In [20]:
accuracy = accuracy_score(
    y_test,
    prediccion
)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9722


## Aplicamos la matriz de confusión 

In [21]:
cm = confusion_matrix(
    y_test,
    prediccion
)

cm_df = pd.DataFrame(
    cm,
    index=modelo.classes_,
    columns=modelo.classes_
)

cm_df

,0,1,2
0,12,0,0
1,1,13,0
2,0,0,10


## Aquí definiremos el reporte de clasificación 

In [22]:
print(
    classification_report(
        y_test,
        prediccion
    )
)

              precision    recall  f1-score   support

           0       0.92      1.00      0.96        12
           1       1.00      0.93      0.96        14
           2       1.00      1.00      1.00        10

    accuracy                           0.97        36
   macro avg       0.97      0.98      0.97        36
weighted avg       0.97      0.97      0.97        36



# Conclusiones
## Limpieza de datos
Se verificó la existencia de valores nulos mediante isnull().sum().
El conjunto de datos Wine no contiene valores faltantes, por lo que no fue necesario eliminar registros ni realizar imputaciones.
La variable objetivo fue renombrada de target a clase para facilitar su interpretación y posteriormente se reemplazaron los valores numéricos por nombres descriptivos.
Debido a que el algoritmo KNN utiliza la distancia entre observaciones, fue necesario aplicar StandardScaler para estandarizar todas las variables y evitar que aquellas con mayor magnitud influyeran más en la clasificación.
## Hallazgos
El modelo KNN fue entrenado utilizando el 80 % de los datos y evaluado con el 20 % restante.
Se utilizó un valor de k = 3, suficiente para obtener una buena clasificación del conjunto Wine.
La estandarización permitió que todas las características tuvieran la misma importancia durante el cálculo de distancias.
La matriz de confusión mostró cuántas muestras fueron clasificadas correctamente y cuáles presentaron errores.